# 🛒 Olist E-Commerce — SQL Analytics

A self-contained analysis of the **Olist Brazilian E-Commerce** dataset (≈100K orders, 96K customers, 3,095 sellers, Sep 2016 – Oct 2018), running on **SQLite inside Colab** — no install, no database server.

This notebook walks through the analysis end to end: it loads the data, builds an analytical foundation, and answers a sequence of business questions about **revenue, operations, retention, and customer satisfaction** — with every figure computed directly from the raw tables.

## The approach (read once)

A few design choices keep every number consistent and trustworthy:

- **Canonical identity.** A customer is identified by `customer_unique_id`, not `customer_id` (which is assigned per order). Using the right key matters for any customer-level metric.
- **Order-grain revenue.** The `order_items` table has one row per line item, so revenue is first rolled up to one row per order. This prevents multi-item orders from inflating counts.
- **Delivered-orders scope.** Core KPIs use *delivered* orders only — realized revenue and behaviour — and this scope is stated explicitly.
- **Two revenue definitions.** `merchandise` (item price) and `gross_billings` (price + freight) are both exposed, so each question uses the right one.
- **Reconciliation.** GMV is tied out against actual customer payments, so the totals are verified rather than assumed.

## How to run
1. Click the **folder icon** in the left sidebar to open the file panel.
2. **Drag all 8 CSVs** into it (`customers, orders, order_items, order_payments, order_reviews, sellers, products, product_category_name_translation`).
3. **Runtime → Run all.**


## 1 · Setup — load the data and build indexes

Each CSV becomes a SQLite table. Indexes on the join and filter columns keep the heavier queries (cohorts, seller aggregates) fast.

In [ ]:
import pandas as pd, sqlite3, os, re
from IPython.display import display

con = sqlite3.connect("olist.db")

TABLES = {
    "orders":               "orders.csv",
    "order_items":          "order_items.csv",
    "order_payments":       "order_payments.csv",
    "order_reviews":        "order_reviews.csv",
    "customers":            "customers.csv",
    "sellers":              "sellers.csv",
    "products":             "products.csv",
    "category_translation": "product_category_name_translation.csv",
}

for table, fname in TABLES.items():
    path = f"/content/{fname}"
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {fname} — drag it into the file panel on the left.")
    df = pd.read_csv(path)
    df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]   # strip any byte-order mark
    df.to_sql(table, con, if_exists="replace", index=False)
    print(f"{table:22s} {len(df):>7,} rows")

# Indexes on the columns used in joins and filters
for idx in [
    "CREATE INDEX IF NOT EXISTS ix_oi_order   ON order_items(order_id)",
    "CREATE INDEX IF NOT EXISTS ix_oi_seller  ON order_items(seller_id)",
    "CREATE INDEX IF NOT EXISTS ix_oi_product ON order_items(product_id)",
    "CREATE INDEX IF NOT EXISTS ix_o_cust     ON orders(customer_id)",
    "CREATE INDEX IF NOT EXISTS ix_o_status   ON orders(order_status)",
    "CREATE INDEX IF NOT EXISTS ix_o_purch    ON orders(order_purchase_timestamp)",
    "CREATE INDEX IF NOT EXISTS ix_c_id       ON customers(customer_id)",
    "CREATE INDEX IF NOT EXISTS ix_pay_order  ON order_payments(order_id)",
    "CREATE INDEX IF NOT EXISTS ix_rev_order  ON order_reviews(order_id)",
]:
    con.execute(idx)
con.commit()
print("\nLoaded and indexed — ready.")


### A small helper

`run_sql()` executes a SQL string and renders each result as a table. View/DDL statements run quietly; every `SELECT` is displayed.

In [ ]:
def run_sql(sql_text, con=con):
    sql = re.sub(r"/\*.*?\*/", "", sql_text, flags=re.S)   # strip /* block */ comments
    sql = re.sub(r"--[^\n]*", "", sql)                       # strip -- line comments
    for stmt in [s.strip() for s in sql.split(";") if s.strip()]:
        if stmt.upper().lstrip().startswith(("CREATE", "DROP")):
            con.execute(stmt); con.commit()
            print("✓ view ready")
        else:
            display(pd.read_sql_query(stmt, con))


## 2 · Canonical views — the analytical foundation

Three views are created once and reused everywhere:

- **`v_delivered_orders`** — delivered orders joined to their customer (`customer_unique_id`). This is the "realized order" universe.
- **`v_order_revenue`** — revenue collapsed to **one row per order**, so an order with five line items still counts as one order. Exposes both `merchandise` (price) and `gross_billings` (price + freight).
- **`v_delivered_revenue`** — delivered orders enriched with their order-level revenue; the view every GMV/AOV query reads from.

Defining scope and grain in one place is what keeps the rest of the numbers internally consistent.

In [ ]:
sql = r"""
-- ============================================================
-- Canonical views — the analytical foundation (run first)
-- Defining scope and revenue grain once keeps every metric below
-- consistent. customer_unique_id is the stable customer identity;
-- customer_id is assigned per order in this dataset.
-- ============================================================

DROP VIEW IF EXISTS v_delivered_revenue;
DROP VIEW IF EXISTS v_order_revenue;
DROP VIEW IF EXISTS v_delivered_orders;

-- v_delivered_orders : the "realized order" universe (delivered only),
--                      joined to each order's customer identity.
CREATE VIEW v_delivered_orders AS
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    c.customer_state,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered';

-- v_order_revenue : revenue rolled up to ONE row per order, so multi-item
--                   orders never inflate order counts. GMV is exposed both
--                   as merchandise (price) and gross billings (price+freight).
CREATE VIEW v_order_revenue AS
SELECT
    oi.order_id,
    ROUND(SUM(oi.price), 2)                    AS merchandise,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS gross_billings,
    ROUND(SUM(oi.freight_value), 2)            AS freight,
    COUNT(*)                                   AS item_count
FROM order_items oi
GROUP BY oi.order_id;

-- v_delivered_revenue : delivered orders enriched with order-level revenue.
--                       The workhorse for all GMV / AOV reporting.
CREATE VIEW v_delivered_revenue AS
SELECT
    d.order_id, d.customer_unique_id, d.customer_state,
    d.order_purchase_timestamp, d.order_delivered_customer_date,
    r.merchandise, r.gross_billings, r.freight, r.item_count
FROM v_delivered_orders d
JOIN v_order_revenue r ON d.order_id = r.order_id;

"""
run_sql(sql)


## 3 · Data validation & identity

Five contextual checks frame the dataset before any KPI is computed:

1. **Coverage window** — the first and last purchase timestamps.
2. **Order-status composition** — what share of orders is delivered, shipped, canceled, etc. This makes the delivered-only scope an explicit, informed choice.
3. **Customer identity** — `customer_id` is minted *per order*; `customer_unique_id` is the actual person. The gap between the two counts is the volume of repeat identities.
4. **Customer counts** — the canonical base, and separately the narrower set of customers who placed at least one delivered order (a different question).
5. **Referential integrity** — orphaned rows across the key joins should be **0**, which confirms the joins are clean before metrics depend on them.

In [ ]:
sql = r"""
-- ============================================================
-- Data validation & identity  (contextual checks, not KPIs)
-- ============================================================

-- Dataset coverage window
SELECT
    MIN(order_purchase_timestamp) AS first_order,
    MAX(order_purchase_timestamp) AS last_order
FROM orders;

-- Order-status composition
SELECT
    order_status,
    COUNT(*)                                            AS cnt,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)  AS pct_of_orders
FROM orders
GROUP BY order_status
ORDER BY cnt DESC;

-- Customer identity — per-order id vs. the person behind it
SELECT
    (SELECT COUNT(*) FROM orders)                          AS total_orders,
    (SELECT COUNT(DISTINCT customer_id) FROM customers)    AS distinct_customer_id,
    (SELECT COUNT(DISTINCT customer_unique_id) FROM customers) AS distinct_customer_unique_id;

-- Canonical customer count
SELECT COUNT(DISTINCT customer_unique_id) AS real_customers
FROM customers;

-- Customers with at least one delivered order
SELECT COUNT(DISTINCT customer_unique_id) AS customers_with_delivered_order
FROM v_delivered_orders;

-- Referential integrity — orphaned rows should be zero
SELECT
    (SELECT COUNT(*) FROM order_items oi
       LEFT JOIN orders o ON oi.order_id = o.order_id
      WHERE o.order_id IS NULL)                       AS items_without_order,
    (SELECT COUNT(*) FROM orders o
       LEFT JOIN customers c ON o.customer_id = c.customer_id
      WHERE c.customer_id IS NULL)                    AS orders_without_customer,
    (SELECT COUNT(*) FROM order_items oi
       LEFT JOIN sellers s ON oi.seller_id = s.seller_id
      WHERE s.seller_id IS NULL)                      AS items_without_seller;

"""
run_sql(sql)


## 4 · Core business KPIs — monthly GMV & AOV

The headline revenue series, on delivered orders, bucketed by **purchase month**:

- **Merchandise GMV** = `SUM(price)` (excludes freight)
- **Gross billings** = `SUM(price + freight)` (what the buyer actually paid)
- **AOV** = merchandise GMV ÷ delivered orders

A **recursive calendar** generates one row per month so the time series has no gaps, and orders are matched to each month on a half-open **date range** — which keeps the timestamp column index-friendly.

*Reading it:* expect steady growth from late-2016 to ~R$1M/month through 2018, on a stable AOV around R$135. Months with zero delivered orders show a blank AOV.

In [ ]:
sql = r"""
-- ============================================================
-- Core KPIs — monthly revenue & unit economics
--   Merchandise GMV = SUM(price)         (excludes freight)
--   Gross billings  = SUM(price+freight) (what the buyer paid)
--   AOV             = merchandise GMV / delivered orders
-- A recursive calendar yields one row per month (no gaps), and orders
-- are matched to each month on a date range (index-friendly).
-- ============================================================

-- Monthly merchandise GMV, gross billings & AOV (gap-free calendar)
WITH RECURSIVE calendar (month_start) AS (
    
    SELECT date(MIN(order_purchase_timestamp), 'start of month') FROM orders
    UNION ALL
    SELECT date(month_start, '+1 month')
    FROM calendar
    WHERE month_start < (
        SELECT date(MAX(order_purchase_timestamp), 'start of month') FROM orders
    )
)

SELECT
    strftime('%Y-%m', cal.month_start)                AS order_month,
    COUNT(dr.order_id)                                AS delivered_orders,
    ROUND(COALESCE(SUM(dr.merchandise), 0), 2)        AS merchandise_gmv,
    ROUND(COALESCE(SUM(dr.gross_billings), 0), 2)     AS gross_billings,
    ROUND(
        COALESCE(SUM(dr.merchandise), 0)
        / NULLIF(COUNT(dr.order_id), 0),
        2
    )                                                 AS aov
FROM calendar cal
LEFT JOIN v_delivered_revenue dr
       ON dr.order_purchase_timestamp >= cal.month_start
      AND dr.order_purchase_timestamp <  date(cal.month_start, '+1 month')
GROUP BY cal.month_start
ORDER BY cal.month_start;

"""
run_sql(sql)


## 5 · Operations — cycle time & cancellation rate

Two measures of post-purchase execution:

- **Cycle time** — average days from purchase to delivery, reported alongside a count of delivered orders missing a delivery date (a data-completeness signal) and how far ahead of the estimated date orders typically arrive.
- **Cancellation rate** — framed as a single coherent cohort: *of all orders **placed** in a month, what share were ultimately canceled.* One cohort, one denominator. The most recent months, too new to have finalised, are flagged **`CENSORED`** so they aren't read as mature.

*Reading it:* cancellation runs well under 1% in mature months; the `CENSORED` tail is expected and should be interpreted with care.

In [ ]:
sql = r"""
-- ============================================================
-- Operations — fulfillment cycle time & cancellation rate
-- ============================================================

-- Order cycle time + delivery-date completeness
SELECT
    COUNT(*)                                                       AS delivered_orders,
    SUM(CASE WHEN order_delivered_customer_date IS NULL
              OR order_delivered_customer_date = '' THEN 1 ELSE 0 END)
                                                                   AS missing_delivery_date,
    ROUND(
        AVG(
            CASE WHEN order_delivered_customer_date IS NOT NULL
                  AND order_delivered_customer_date <> ''
                 THEN julianday(order_delivered_customer_date)
                    - julianday(order_purchase_timestamp)
            END
        ), 2
    )                                                              AS avg_cycle_days,
    ROUND(
        AVG(
            CASE WHEN order_delivered_customer_date IS NOT NULL
                  AND order_delivered_customer_date <> ''
                 THEN julianday(order_estimated_delivery_date)
                    - julianday(order_delivered_customer_date)
            END
        ), 2
    )                                                              AS avg_days_early_vs_estimate
FROM v_delivered_orders;

-- Monthly cancellation rate — single purchase cohort, censor-aware
WITH bounds AS (
    
    SELECT strftime('%Y-%m',
             date(MAX(order_purchase_timestamp), 'start of month', '-1 month')
           ) AS censor_from
    FROM orders
),
monthly AS (
    SELECT
        strftime('%Y-%m', order_purchase_timestamp)                          AS purchase_month,
        COUNT(*)                                                             AS orders_placed,
        SUM(CASE WHEN order_status = 'canceled'  THEN 1 ELSE 0 END)          AS canceled,
        SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END)          AS delivered,
        SUM(CASE WHEN order_status NOT IN ('delivered','canceled')
                 THEN 1 ELSE 0 END)                                          AS still_open
    FROM orders
    GROUP BY purchase_month
)
SELECT
    m.purchase_month,
    m.orders_placed,
    m.delivered,
    m.canceled,
    m.still_open,
    ROUND(100.0 * m.canceled / m.orders_placed, 2)                           AS cancel_rate_pct,
    CASE WHEN m.purchase_month >= (SELECT censor_from FROM bounds)
         THEN 'CENSORED' ELSE 'mature' END                                   AS cohort_status
FROM monthly m
ORDER BY m.purchase_month;

"""
run_sql(sql)


## 6 · Customer analytics — retention & repurchase

Customer-level behaviour, keyed on `customer_unique_id`:

- **Repeat rate** — the share of customers with more than one delivered order.
- **Time to second purchase** — the first→second gap, computed with `LEAD()` in a single pass (no self-join), reported as average, min, and max.
- **Cohort retention curve** — customers grouped by their **first-purchase month**, showing how many return 1, 2, 3, or 4+ months later. A single repeat-rate number hides the *shape* of retention; the curve reveals it.

Window functions order by `(timestamp, order_id)` so results are reproducible run-to-run even when two orders share a timestamp.

*Reading it:* repeat purchasing is thin (~3%) — the curve makes the size of the re-engagement opportunity concrete.

In [ ]:
sql = r"""
-- ============================================================
-- Customer analytics — retention & repurchase behaviour
-- Identity = customer_unique_id. Window functions order by
-- (timestamp, order_id) so results are deterministic when ties occur.
-- ============================================================

-- Repeat-customer rate
SELECT
    COUNT(*)                                                        AS customers,
    SUM(CASE WHEN total_orders > 1 THEN 1 ELSE 0 END)               AS repeat_customers,
    ROUND(100.0 * SUM(CASE WHEN total_orders > 1 THEN 1 ELSE 0 END)
                / COUNT(*), 2)                                      AS repeat_pct
FROM (
    SELECT customer_unique_id, COUNT(DISTINCT order_id) AS total_orders
    FROM v_delivered_orders
    GROUP BY customer_unique_id
);

-- Average time to second purchase (LEAD)
WITH ordered AS (
    SELECT
        customer_unique_id,
        order_purchase_timestamp,
        LEAD(order_purchase_timestamp) OVER (
            PARTITION BY customer_unique_id
            ORDER BY order_purchase_timestamp, order_id
        )                                       AS next_purchase,
        ROW_NUMBER() OVER (
            PARTITION BY customer_unique_id
            ORDER BY order_purchase_timestamp, order_id
        )                                       AS rn
    FROM v_delivered_orders
)
SELECT
    COUNT(*)                                                        AS repeat_customers,
    ROUND(AVG(julianday(next_purchase) - julianday(order_purchase_timestamp)), 2)
                                                                    AS avg_days_to_second_order,
    ROUND(MIN(julianday(next_purchase) - julianday(order_purchase_timestamp)), 2)
                                                                    AS min_days,
    ROUND(MAX(julianday(next_purchase) - julianday(order_purchase_timestamp)), 2)
                                                                    AS max_days
FROM ordered
WHERE rn = 1
  AND next_purchase IS NOT NULL;

-- Acquisition-cohort retention curve
WITH first_order AS (
    SELECT
        customer_unique_id,
        MIN(date(order_purchase_timestamp, 'start of month')) AS cohort_month
    FROM v_delivered_orders
    GROUP BY customer_unique_id
),
activity AS (
    SELECT
        f.cohort_month,
        d.customer_unique_id,
        (  (CAST(strftime('%Y', d.order_purchase_timestamp) AS INT) * 12
            + CAST(strftime('%m', d.order_purchase_timestamp) AS INT))
         - (CAST(strftime('%Y', f.cohort_month) AS INT) * 12
            + CAST(strftime('%m', f.cohort_month) AS INT))
        )                                                     AS month_offset
    FROM v_delivered_orders d
    JOIN first_order f USING (customer_unique_id)
)
SELECT
    strftime('%Y-%m', cohort_month)                                          AS cohort,
    COUNT(DISTINCT CASE WHEN month_offset = 0 THEN customer_unique_id END)   AS m0_acquired,
    COUNT(DISTINCT CASE WHEN month_offset = 1 THEN customer_unique_id END)   AS m1,
    COUNT(DISTINCT CASE WHEN month_offset = 2 THEN customer_unique_id END)   AS m2,
    COUNT(DISTINCT CASE WHEN month_offset = 3 THEN customer_unique_id END)   AS m3,
    COUNT(DISTINCT CASE WHEN month_offset >= 4 THEN customer_unique_id END)  AS m4_plus
FROM activity
GROUP BY cohort_month
ORDER BY cohort_month;

"""
run_sql(sql)


## 7 · Marketplace — geography, sellers & categories

Descriptive context around the two sides of the marketplace:

- **Revenue by customer state**, with freight shown as its own column (a real geographic cost driver).
- **Seller performance** — delivered orders, items sold, items per order, and GMV per order, top 20 by GMV.
- **Top product categories** by GMV, using the translation table for English names.

*Reading it:* demand is geographically concentrated (São Paulo leads by a wide margin), and a handful of categories drive most merchandise value.

In [ ]:
sql = r"""
-- ============================================================
-- Marketplace — geography, sellers & categories (descriptive)
-- ============================================================

-- Revenue by customer state (incl. freight)
SELECT
    customer_state,
    COUNT(DISTINCT order_id)            AS orders,
    ROUND(SUM(merchandise), 2)          AS merchandise_gmv,
    ROUND(SUM(gross_billings), 2)       AS gross_billings,
    ROUND(SUM(freight), 2)              AS freight_cost
FROM v_delivered_revenue
GROUP BY customer_state
ORDER BY merchandise_gmv DESC;

-- Seller performance — volume, complexity, GMV per order
SELECT
    s.seller_id,
    s.seller_state,
    COUNT(DISTINCT oi.order_id)                          AS delivered_orders,
    COUNT(oi.order_item_id)                              AS items_sold,
    ROUND(COUNT(oi.order_item_id) * 1.0
          / COUNT(DISTINCT oi.order_id), 2)              AS items_per_order,
    ROUND(SUM(oi.price), 2)                              AS delivered_gmv,
    ROUND(SUM(oi.price) / COUNT(DISTINCT oi.order_id), 2) AS gmv_per_order
FROM v_delivered_orders d
JOIN order_items oi ON oi.order_id = d.order_id
JOIN sellers s      ON oi.seller_id = s.seller_id
GROUP BY s.seller_id, s.seller_state
ORDER BY delivered_gmv DESC
LIMIT 20;

-- Top product categories by GMV
SELECT
    COALESCE(t.product_category_name_english, p.product_category_name, 'unknown')
                                                AS category,
    COUNT(DISTINCT oi.order_id)                 AS orders,
    COUNT(oi.order_item_id)                     AS items_sold,
    ROUND(SUM(oi.price), 2)                     AS merchandise_gmv
FROM v_delivered_orders d
JOIN order_items oi ON oi.order_id = d.order_id
JOIN products    p  ON oi.product_id = p.product_id
LEFT JOIN category_translation t
       ON p.product_category_name = t.product_category_name
GROUP BY category
ORDER BY merchandise_gmv DESC
LIMIT 15;

"""
run_sql(sql)


## 8 · Payments & reviews — reconciliation and the headline insight

Three analyses that connect the financial and experience sides:

- **Revenue reconciliation** — does item-level GMV tie out to actual customer payments? Comparing `merchandise`, `gross_billings`, and `SUM(payment_value)` shows that payments line up with **price + freight**, with only a tiny residual. This is the check that makes the rest of the numbers trustworthy.
- **Payment-type mix** — method share and average installments. (`payment_value` is the total charged, so installments are a count, not a multiplier.)
- **Delivery speed vs. satisfaction** — the project's headline relationship: average review score and the share of negative reviews (≤2★) across delivery-time buckets.

*Reading it:* satisfaction falls sharply once delivery passes ~30 days — delivery time is the dominant lever on review scores.

In [ ]:
sql = r"""
-- ============================================================
-- Payments & reviews — financial reconciliation + experience
-- ============================================================

-- Revenue reconciliation — GMV vs. customer payments
WITH gmv AS (
    SELECT
        ROUND(SUM(merchandise), 2)     AS merchandise_gmv,
        ROUND(SUM(gross_billings), 2)  AS gross_billings
    FROM v_delivered_revenue
),
pay AS (
    SELECT ROUND(SUM(p.payment_value), 2) AS total_payments
    FROM order_payments p
    JOIN v_delivered_orders d ON p.order_id = d.order_id
)
SELECT
    gmv.merchandise_gmv,
    gmv.gross_billings,
    pay.total_payments,
    ROUND(pay.total_payments - gmv.gross_billings, 2)                       AS residual_vs_billings,
    ROUND(100.0 * (pay.total_payments - gmv.gross_billings)
                / gmv.gross_billings, 4)                                    AS residual_pct
FROM gmv, pay;

-- Payment-type mix & installment behaviour
SELECT
    p.payment_type,
    COUNT(*)                                                AS payment_rows,
    ROUND(SUM(p.payment_value), 2)                          AS total_value,
    ROUND(AVG(p.payment_value), 2)                          AS avg_value,
    ROUND(AVG(p.payment_installments), 2)                   AS avg_installments
FROM order_payments p
JOIN v_delivered_orders d ON p.order_id = d.order_id
GROUP BY p.payment_type
ORDER BY total_value DESC;

-- Delivery speed vs. customer satisfaction
WITH delivery AS (
    SELECT
        order_id,
        julianday(order_delivered_customer_date)
        - julianday(order_purchase_timestamp)               AS cycle_days
    FROM v_delivered_orders
    WHERE order_delivered_customer_date IS NOT NULL
      AND order_delivered_customer_date <> ''
),
scored AS (
    SELECT
        r.review_score,
        CASE
            WHEN d.cycle_days <=  3 THEN '1: 0-3 days'
            WHEN d.cycle_days <=  7 THEN '2: 4-7 days'
            WHEN d.cycle_days <= 14 THEN '3: 8-14 days'
            WHEN d.cycle_days <= 30 THEN '4: 15-30 days'
            ELSE                         '5: 30+ days'
        END                                                 AS speed_bucket
    FROM order_reviews r
    JOIN delivery d ON r.order_id = d.order_id
)
SELECT
    speed_bucket,
    COUNT(*)                                                       AS reviews,
    ROUND(AVG(review_score), 3)                                    AS avg_review_score,
    ROUND(100.0 * SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END)
                / COUNT(*), 2)                                     AS pct_negative
FROM scored
GROUP BY speed_bucket
ORDER BY speed_bucket;

"""
run_sql(sql)


## Summary

The data supports four clear takeaways:

1. **Delivery speed is the dominant driver of satisfaction** — late deliveries collapse review scores.
2. **Retention is the growth ceiling** — only ~3% of customers reorder.
3. **Revenue is geographically concentrated** — a few states drive most GMV.
4. **The revenue model reconciles** — GMV ties out to payments once freight is included.

Every figure above was computed directly from the raw tables in this notebook.